# Characterizing the customer

In [1]:
## python built-in modules
from pathlib import Path

## third party modules
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt

## loads specific package modules, which contain project settings
from client_segmentation.config import PROCESSED_DATA_DIR
from client_segmentation.config import RAW_DATA_DIR
from client_segmentation.config import INTERIM_DATA_DIR

2025-04-08 15:56:52.465 | INFO     | client_segmentation.config:<module>:11 - PROJ_ROOT path is: /home/gasobral/Meus Arquivos/data-science/client_segmentation


In [2]:
## below there is a mapping of dtypes for every column of each csv file
## data_mapping is a dictionary which its key = csv file and
## value = dict (dtype mapping, where key = column and value = dtype)
product_category_name_translation_map = {
    'product_category_name' : 'string',
    'product_category_name_english' : 'string'
}

olist_sellers_dataset_map = {
    'seller_id': 'string',
    'seller_zip_code_prefix': 'int64',
    'seller_city': 'string',
    'seller_state': 'string'
}

olist_geolocation_dataset_map = {
    'geolocation_zip_code_prefix': 'int64',
    'geolocation_lat': 'float64',
    'geolocation_lng': 'float64',
    'geolocation_city': 'string',
    'geolocation_state': 'string'
}

olist_products_dataset_map = {
    'product_id': 'string',
    'product_category_name': 'string',
    'product_name_lenght': 'float64',
    'product_description_lenght': 'float64',
    'product_photos_qty':  'float64',
    'product_weight_g': 'float64',
    'product_length_cm': 'float64',
    'product_height_cm': 'float64',
    'product_width_cm': 'float64'
}

olist_order_items_dataset_map = {
    'order_id': 'string',
    'order_item_id': 'int64',
    'product_id': 'string',
    'seller_id': 'string',
    'shipping_limit_date': 'object',
    'price': 'float64',
    'freight_value': 'float64'
}

olist_order_payments_dataset_map = {
    'order_id': 'string',
    'payment_sequential': 'int64',
    'payment_type': 'string',
    'payment_installments': 'int64',
    'payment_value': 'float64',
}

olist_orders_dataset_map = {
    'order_id': 'string',
    'customer_id': 'string',
    'order_status': 'string',
    'order_purchase_timestamp': 'object',
    'order_approved_at': 'object',
    'order_delivered_carrier_date': 'object',
    'order_delivered_customer_date': 'object',
    'order_estimated_delivery_date': 'object'
}

olist_order_reviews_dataset_map = {
    'review_id': 'string',
    'order_id': 'string',
    'review_score': 'int64',
    'review_comment_title': 'string',
    'review_comment_message': 'string',
    'review_creation_date': 'object',
    'review_answer_timestamp': 'object'
}

olist_customers_dataset_map = {
     'customer_id': 'string',
     'customer_unique_id': 'string',
     'customer_zip_code_prefix': 'int64',
     'customer_city': 'string',
     'customer_state': 'string'
}

data_mapping = {
    'product_category_name_translation': product_category_name_translation_map,
    'olist_sellers_dataset': olist_sellers_dataset_map,
    'olist_geolocation_dataset': olist_geolocation_dataset_map,
    'olist_products_dataset': olist_products_dataset_map,
    'olist_order_items_dataset': olist_order_items_dataset_map,
    'olist_order_payments_dataset': olist_order_payments_dataset_map,
    'olist_orders_dataset': olist_orders_dataset_map,
    'olist_order_reviews_dataset': olist_order_reviews_dataset_map,
    'olist_customers_dataset': olist_customers_dataset_map
}

def data_aquisition(file_path: Path,
                    data_mapping: dict) -> pd.DataFrame:
    """
    Given a file path, reads a dataset and return a data frame.
    
    Arguments
    ---------
    file_path: a file path for a csv file.
    
    data_mapping: a dictionary with a mapping, which tells type of data
                  for each column of csv file.

    Return
    ------
    A data frame with the data loaded from csv file.
    """

    ## create file without file extention (suffix) in order to
    ## obtain the date mapping for dataset columns 
    name_suffixless = file_path.name.split('.')[0]
    dataset = pd.read_csv(file_path,
                          dtype=data_mapping[name_suffixless],
                          date_format="%Y-%m-%d %H:%M:%S")

    ## creates a list only with the columns which contain data on
    ## their name
    date_columns = [col for col in dataset.columns
                    if "date" in col or
                    "time" in col or
                    "order_approved_at" in col]

    if len(date_columns) != 0:
        for col in date_columns:
            dataset[col] = pd.to_datetime(dataset[col],
                                          format="%Y-%m-%d %H:%M:%S")
    else:
        print("Dataset has no columns which contain date!"
              " No parsing required.")

    return dataset

## Analyzing customer data

In [4]:
customer = data_aquisition(INTERIM_DATA_DIR / 'olist_customers_dataset.csv',
                           data_mapping)

Dataset has no columns which contain date! No parsing required.


In [5]:
customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  string
 1   customer_unique_id        99441 non-null  string
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  string
 4   customer_state            99441 non-null  string
dtypes: int64(1), string(4)
memory usage: 3.8 MB


In [7]:
customer.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


For customer specific data, we only its location data (city and state),
we do not have their names or any personal information. Now let's check
which kind of stuff a customer buys/orders.

Regarding data schema shown in
[Olist dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce),
product specific data is located at olist_product_dataset. Moreover, in
order to know which products a customer order, we must join the datasets
olist_order_customer_dataset, olist_orders_dataset,
olist_order_itens_dataset and olist_produts_dataset.

In [13]:
order = data_aquisition(INTERIM_DATA_DIR / 'olist_orders_dataset.csv',
                        data_mapping)

order_items = data_aquisition(INTERIM_DATA_DIR / 'olist_order_items_dataset.csv',
                              data_mapping)

products = data_aquisition(INTERIM_DATA_DIR / 'olist_products_dataset.csv',
                           data_mapping)

Dataset has no columns which contain date! No parsing required.


In [17]:
customer_order = pd.merge(customer,
                          order,
                          how='inner',
                          on='customer_id')

order_products = pd.merge(order_items,
                          products,
                          how='inner',
                          on='product_id')

customer_products = pd.merge(customer_order,
                             order_products,
                             how='inner',
                             on='order_id')

In [18]:
customer_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 26 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   customer_id                    112650 non-null  string        
 1   customer_unique_id             112650 non-null  string        
 2   customer_zip_code_prefix       112650 non-null  int64         
 3   customer_city                  112650 non-null  string        
 4   customer_state                 112650 non-null  string        
 5   order_id                       112650 non-null  string        
 6   order_status                   112650 non-null  string        
 7   order_purchase_timestamp       112650 non-null  datetime64[ns]
 8   order_approved_at              112635 non-null  datetime64[ns]
 9   order_delivered_carrier_date   111456 non-null  datetime64[ns]
 10  order_delivered_customer_date  110196 non-null  datetime64[ns]
 11  

In [19]:
customer_products.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,...,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,...,124.99,21.88,moveis_escritorio,41.0,1141.0,1.0,8683.0,54.0,64.0,31.0
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,...,289.00,46.48,utilidades_domesticas,43.0,1002.0,3.0,10150.0,89.0,15.0,40.0
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,...,139.94,17.79,moveis_escritorio,55.0,955.0,1.0,8267.0,52.0,52.0,17.0
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,...,149.94,23.36,moveis_escritorio,48.0,1066.0,1.0,12160.0,56.0,51.0,28.0
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,...,230.00,22.25,casa_conforto,61.0,407.0,1.0,5200.0,45.0,15.0,35.0


In [30]:
category_counts = customer_products['product_category_name'].value_counts()

## shows the top 10 most ordered categories
print(category_counts.head(10), end='\n\n')

## also shows the porcentage of the top 10 most order categoires
print(category_counts.head(10) / customer_products.shape[0] * 100)

product_category_name
cama_mesa_banho           11115
beleza_saude               9670
esporte_lazer              8641
moveis_decoracao           8334
informatica_acessorios     7827
utilidades_domesticas      6964
relogios_presentes         5991
telefonia                  4545
ferramentas_jardim         4347
automotivo                 4235
Name: count, dtype: Int64

product_category_name
cama_mesa_banho           9.866844
beleza_saude               8.58411
esporte_lazer             7.670661
moveis_decoracao          7.398136
informatica_acessorios    6.948069
utilidades_domesticas      6.18198
relogios_presentes        5.318242
telefonia                 4.034621
ferramentas_jardim        3.858855
automotivo                3.759432
Name: count, dtype: Float64


In [ ]:
## categoria por valor
## horário das ordens (+ por UF)